# PayShield AI — Feature Engineering

## AI-Powered Payment Success Optimization

### Objective

Transform raw payment transactions into time-window-based
monitoring features that can be used by machine learning models.

We will create features such as:

- Transaction volume
- Payment failure rate
- Timeout rate
- Average latency
- Maximum latency
- 95th percentile latency
- Bank error rate
- Transaction amount statistics
- Time-based features

The resulting dataset will represent the health of each
receiver bank over time.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/transactions.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (100000, 15)


In [3]:
df.head()

,transaction_id,timestamp,amount,sender_bank,receiver_bank,payment_method,upi_app,bank_health,sender_bank_health,gateway_health,latency_ms,payment_status,error_code,timeout,root_cause
0,TXN0000001,2026-01-01 00:00:37,472.69,AXIS,CANARA,UPI,PhonePe,NORMAL,NORMAL,NORMAL,604.531470,SUCCESS,NONE,0,NORMAL
1,TXN0000002,2026-01-01 00:00:56,1287.24,PNB,PNB,UPI,Paytm,NORMAL,NORMAL,NORMAL,1197.074588,SUCCESS,NONE,0,NORMAL
2,TXN0000003,2026-01-01 00:00:59,587.20,HDFC,HDFC,UPI,GooglePay,NORMAL,NORMAL,NORMAL,1271.020534,SUCCESS,NONE,0,NORMAL
3,TXN0000004,2026-01-01 00:01:09,145.57,BOB,PNB,UPI,PhonePe,NORMAL,NORMAL,NORMAL,957.377173,SUCCESS,NONE,0,NORMAL
4,TXN0000005,2026-01-01 00:01:13,574.99,INDUSIND,HDFC,UPI,GooglePay,NORMAL,NORMAL,NORMAL,1165.837966,SUCCESS,NONE,0,NORMAL


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   transaction_id      100000 non-null  object        
 1   timestamp           100000 non-null  datetime64[ns]
 2   amount              100000 non-null  float64       
 3   sender_bank         100000 non-null  object        
 4   receiver_bank       100000 non-null  object        
 5   payment_method      100000 non-null  object        
 6   upi_app             100000 non-null  object        
 7   bank_health         100000 non-null  object        
 8   sender_bank_health  100000 non-null  object        
 9   gateway_health      100000 non-null  object        
 10  latency_ms          100000 non-null  float64       
 11  payment_status      100000 non-null  object        
 12  error_code          100000 non-null  object        
 13  timeout             100000 non

In [5]:
df["time_window"] = (
    df["timestamp"]
    .dt.floor("5min")
)

df[[
    "timestamp",
    "time_window"
]].head(10)

,timestamp,time_window
0,2026-01-01 00:00:37,2026-01-01
1,2026-01-01 00:00:56,2026-01-01
2,2026-01-01 00:00:59,2026-01-01
3,2026-01-01 00:01:09,2026-01-01
4,2026-01-01 00:01:13,2026-01-01
5,2026-01-01 00:01:56,2026-01-01
6,2026-01-01 00:02:06,2026-01-01
7,2026-01-01 00:02:13,2026-01-01
8,2026-01-01 00:02:39,2026-01-01
9,2026-01-01 00:03:01,2026-01-01


In [6]:
df["is_failed"] = (
    df["payment_status"] == "FAILED"
).astype(int)

df["is_timeout"] = (
    df["timeout"] == 1
).astype(int)

df["is_bank_error"] = (
    df["error_code"].isin([
        "BANK_TIMEOUT",
        "BANK_SERVER_ERROR"
    ])
).astype(int)

In [7]:
grouped = df.groupby([
    "receiver_bank",
    "time_window"
])

In [8]:
monitoring = (
    grouped
    .agg(
        transaction_count=(
            "transaction_id",
            "count"
        )
    )
    .reset_index()
)

monitoring.head()

,receiver_bank,time_window,transaction_count
0,AXIS,2026-01-01 00:00:00,2
1,AXIS,2026-01-01 00:05:00,2
2,AXIS,2026-01-01 00:20:00,4
3,AXIS,2026-01-01 00:35:00,2
4,AXIS,2026-01-01 00:45:00,3


In [9]:
failure_features = (
    grouped
    .agg(
        failure_rate=(
            "is_failed",
            "mean"
        )
    )
    .reset_index()
)

failure_features["failure_rate"] *= 100

failure_features.head()

,receiver_bank,time_window,failure_rate
0,AXIS,2026-01-01 00:00:00,0.0
1,AXIS,2026-01-01 00:05:00,0.0
2,AXIS,2026-01-01 00:20:00,0.0
3,AXIS,2026-01-01 00:35:00,0.0
4,AXIS,2026-01-01 00:45:00,0.0


In [10]:
timeout_features = (
    grouped
    .agg(
        timeout_rate=(
            "is_timeout",
            "mean"
        )
    )
    .reset_index()
)

timeout_features["timeout_rate"] *= 100

timeout_features.head()

,receiver_bank,time_window,timeout_rate
0,AXIS,2026-01-01 00:00:00,0.0
1,AXIS,2026-01-01 00:05:00,0.0
2,AXIS,2026-01-01 00:20:00,0.0
3,AXIS,2026-01-01 00:35:00,0.0
4,AXIS,2026-01-01 00:45:00,0.0


In [11]:
latency_features = (
    grouped["latency_ms"]
    .agg(
        avg_latency="mean",
        max_latency="max",
        p95_latency=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

latency_features.head()

,receiver_bank,time_window,avg_latency,max_latency,p95_latency
0,AXIS,2026-01-01 00:00:00,792.636797,1001.694611,980.788829
1,AXIS,2026-01-01 00:05:00,1373.890105,1758.332274,1719.888057
2,AXIS,2026-01-01 00:20:00,1415.224907,1743.482140,1721.421629
3,AXIS,2026-01-01 00:35:00,1076.585366,1096.005956,1094.063897
4,AXIS,2026-01-01 00:45:00,1316.782330,1657.078125,1617.411614


In [12]:
error_features = (
    grouped
    .agg(
        bank_error_rate=(
            "is_bank_error",
            "mean"
        )
    )
    .reset_index()
)

error_features["bank_error_rate"] *= 100

error_features.head()

,receiver_bank,time_window,bank_error_rate
0,AXIS,2026-01-01 00:00:00,0.0
1,AXIS,2026-01-01 00:05:00,0.0
2,AXIS,2026-01-01 00:20:00,0.0
3,AXIS,2026-01-01 00:35:00,0.0
4,AXIS,2026-01-01 00:45:00,0.0


In [13]:
grouped = df.groupby(
    ["receiver_bank", "time_window"]
)

print("Grouping created successfully!")

Grouping created successfully!


In [14]:
amount_features = (
    df.groupby(["receiver_bank", "time_window"])["amount"]
    .agg(
        avg_amount="mean",
        max_amount="max"
    )
    .reset_index()
)

print("Amount features created successfully!")
amount_features.head()

Amount features created successfully!


,receiver_bank,time_window,avg_amount,max_amount
0,AXIS,2026-01-01 00:00:00,413.660,586.93
1,AXIS,2026-01-01 00:05:00,487.480,567.26
2,AXIS,2026-01-01 00:20:00,365.135,594.50
3,AXIS,2026-01-01 00:35:00,777.685,1512.16
4,AXIS,2026-01-01 00:45:00,289.110,521.04


In [15]:
print(amount_features.columns)

Index(['receiver_bank', 'time_window', 'avg_amount', 'max_amount'], dtype='object')


In [16]:
monitoring = (
    monitoring
    .merge(
        failure_features,
        on=["receiver_bank", "time_window"]
    )
    .merge(
        timeout_features,
        on=["receiver_bank", "time_window"]
    )
    .merge(
        latency_features,
        on=["receiver_bank", "time_window"]
    )
    .merge(
        error_features,
        on=["receiver_bank", "time_window"]
    )
    .merge(
        amount_features,
        on=["receiver_bank", "time_window"]
    )
)

In [17]:
monitoring.head()

,receiver_bank,time_window,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount
0,AXIS,2026-01-01 00:00:00,2,0.0,0.0,792.636797,1001.694611,980.788829,0.0,413.660,586.93
1,AXIS,2026-01-01 00:05:00,2,0.0,0.0,1373.890105,1758.332274,1719.888057,0.0,487.480,567.26
2,AXIS,2026-01-01 00:20:00,4,0.0,0.0,1415.224907,1743.482140,1721.421629,0.0,365.135,594.50
3,AXIS,2026-01-01 00:35:00,2,0.0,0.0,1076.585366,1096.005956,1094.063897,0.0,777.685,1512.16
4,AXIS,2026-01-01 00:45:00,3,0.0,0.0,1316.782330,1657.078125,1617.411614,0.0,289.110,521.04


In [18]:
monitoring["hour"] = (
    monitoring["time_window"].dt.hour
)

monitoring["day_of_week"] = (
    monitoring["time_window"].dt.dayofweek
)

monitoring["is_weekend"] = (
    monitoring["day_of_week"] >= 5
).astype(int)

In [19]:
health_features = (
    grouped["bank_health"]
    .agg(
        bank_health=lambda x: x.mode()[0]
    )
    .reset_index()
)

health_features.head()

,receiver_bank,time_window,bank_health
0,AXIS,2026-01-01 00:00:00,NORMAL
1,AXIS,2026-01-01 00:05:00,NORMAL
2,AXIS,2026-01-01 00:20:00,NORMAL
3,AXIS,2026-01-01 00:35:00,NORMAL
4,AXIS,2026-01-01 00:45:00,NORMAL


In [20]:
monitoring = monitoring.merge(
    health_features,
    on=["receiver_bank", "time_window"]
)

In [21]:
print("Monitoring dataset shape:")
print(monitoring.shape)

print("\nColumns:")
print(monitoring.columns.tolist())

Monitoring dataset shape:
(59266, 15)

Columns:
['receiver_bank', 'time_window', 'transaction_count', 'failure_rate', 'timeout_rate', 'avg_latency', 'max_latency', 'p95_latency', 'bank_error_rate', 'avg_amount', 'max_amount', 'hour', 'day_of_week', 'is_weekend', 'bank_health']


In [22]:
print(
    monitoring["bank_health"].value_counts()
)

bank_health
NORMAL      58930
SEVERE        162
DEGRADED       94
RECOVERY       80
Name: count, dtype: int64


In [23]:
monitoring.groupby("bank_health")[
    [
        "transaction_count",
        "failure_rate",
        "timeout_rate",
        "avg_latency",
        "p95_latency",
        "bank_error_rate"
    ]
].mean()

,transaction_count,failure_rate,timeout_rate,avg_latency,p95_latency,bank_error_rate
bank_health,,,,,,
DEGRADED,1.574468,30.336879,68.120567,3589.359068,3832.212214,30.336879
NORMAL,1.687273,2.555932,0.774688,1231.080646,1323.822609,1.055457
RECOVERY,1.725000,15.458333,3.125000,1926.918636,2088.818556,4.375000
SEVERE,1.746914,58.991770,90.792181,5265.514214,5816.851666,58.991770


In [24]:
monitoring.to_csv(
    "../data/processed/payment_monitoring_features.csv",
    index=False
)

print(
    "Feature dataset saved successfully!"
)

Feature dataset saved successfully!
